# 🎬 WE4 · Notebook 01 — Reinforcement Learning, from scratch
## Learning how to act, in an interview you are in no way qualified for

> **The story.** You graduated last month. Unfortunately, four years of letting a chatbot do the
> thinking have left your brain with the structural integrity of a wet croissant, and in eleven
> minutes you have a technical interview at **GaGGle** — best pay in the industry, best research in
> the world, and a hiring process with a reputation.
>
> You will not learn machine learning before 11 a.m. But two things are in your favour. The
> interviewer is of the older generation and their hearing and eyesight are, let us say, *generous*.
> And you own an unreasonable number of costumes — so when an interview goes badly, you put on a
> moustache and get back in the queue.
>
> You cannot learn the material. You can learn **how to act**.

That sentence is not a joke, it is the definition. If reinforcement learning had to fit on one line
it would be:

> ### **“Learning how to act, given the state I am in.”**

Everything in this notebook — and in the four that follow it this week — is bookkeeping around that
sentence. By the end of the hour you will have taken it apart into its five pieces, written down
what *good* even means, and learned a table of behaviour from nothing but repeated humiliation.

**How this notebook works**
- Short explanations, then small hands-on tasks marked **🎯** for you to fill in.
- **Interactive widgets** to play with each idea *before* the maths shows up.
- The environment is a person, and this notebook deliberately **refuses to show you their brain**.
  You will never see a transition probability. That restriction is not a simplification — it is the
  entire reason the algorithms of this week look the way they do.
- ⏱️ Budget about an hour. There is more reading than typing, on purpose.

## 0. Setup

This notebook is **self-contained**: the first cell pulls the exercise files (the `intro_viz.py`
display helpers, which also happen to contain the interviewer) directly from the course repository.
Run the setup cells below in order.

**0.1 — Fetch the exercise files.**

In [ ]:
import os, sys

REPO_OWNER = "eth-fdd-fs26"
REPO_NAME  = "FDD-WE4-public"
HELPER     = os.path.join("1_rl_intro", "exercise", "intro_viz.py")

def _in_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except Exception:
        return False

if _in_colab():
    url = f"https://github.com/{REPO_OWNER}/{REPO_NAME}.git"
    if not os.path.isdir(REPO_NAME):
        print("Cloning the exercise repo…")
        !git clone -q "$url"
    else:                                 # already cloned earlier — refresh to the latest version
        print("Updating the exercise repo to the latest version…")
        !git -C "$REPO_NAME" pull -q "$url" || echo "  (could not pull — using the existing copy)"

# Move to the REPO ROOT — the folder holding `1_rl_intro/exercise/` — so imports resolve cleanly.
for _root in [REPO_NAME, ".", os.path.dirname(os.getcwd()),
              os.path.dirname(os.path.dirname(os.getcwd())), os.getcwd()]:
    if os.path.exists(os.path.join(_root, HELPER)):
        os.chdir(_root)
        break
else:
    raise FileNotFoundError(
        "Could not find the repo (1_rl_intro/exercise/intro_viz.py). If it is still private, add a "
        "GITHUB_TOKEN secret (see the note above) and re-run this cell.")
sys.path.insert(0, os.path.join(os.getcwd(), "1_rl_intro", "exercise"))   # make helpers importable
print("Working directory:", os.getcwd())

**0.2 — Install dependencies.** All of these are already on Colab; this just pins versions
(and makes the notebook work outside Colab too).

In [ ]:
%pip install -q -r 1_rl_intro/exercise/requirements_intro.txt

**0.3 — Import the libraries.** The scene, the widgets and the quizzes live in **`intro_viz`**
so the teaching cells stay about the *idea*, not about HTML. The interviewer lives in there too, and
you will only ever reach them through `reset()` and `step()` — never through their probabilities.

In [ ]:
import numpy as np
from collections import defaultdict

import importlib
import intro_viz as iv
importlib.reload(iv)       # pick up the latest helpers even if a stale copy was cached

np.random.seed(0)
np.set_printoptions(precision=3, suppress=True)
print("Environment ready ✅  ·  it is 10:58 and you are about to be asked about transformers")

---
# Part 1 — The five words

Reinforcement learning is described by five objects. Here they are, with the column that usually
gets left out: **who chooses each one.**

In [ ]:
iv.five_words()

Let's start with the two that are simply handed to you.

In [ ]:
iv.the_setup()

## 1.1 · Action — the complete list of things you can do

An **action** is a move the agent can make. That is the whole definition. What matters is that the
list is *closed*: whatever happens in that room, you will be choosing one of exactly four things.

You cannot answer the question. You cannot explain the attention mechanism. Your action space is:

In [ ]:
ACTIONS      = iv.ACTIONS         # ['Smile', 'Jargon', 'Compliment', 'Agree hard']
ACTION_EMOJI = iv.ACTION_EMOJI
N_ACTIONS    = len(ACTIONS)

for a in range(N_ACTIONS):
    print(f"  action {a} = {ACTION_EMOJI[a]}  {ACTIONS[a]}")
print(f"\n{N_ACTIONS} actions. Every policy in this notebook is, ultimately, a way of picking "
      f"one of these numbers.")

> 💡 **Actions are what you *do*, not what you *achieve*.** “Get hired” is not an action. “Make
> them like me” is not an action. `2` is an action.

## 1.2 · Reward — the only thing that counts as success

The **reward** `r` is a number the world hands you after every action, and the agent's entire
existence is organised around collecting it. Here it is almost insultingly simple:

| what happens | reward |
|---|---|
| 🥳 “You are taken.” | **+1**, and the interview ends |
| 💔 “…move on to the next candidate.” | **−1**, and the interview ends |
| anything else they say | **0**, and the interview continues |

This is a **sparse reward**: zero, zero, zero, zero, and then one number at the very end. Notice
what that means — for most of the interview, *nothing tells you whether you are doing well*. The
turn where you said the perfect thing and the turn where you nearly blew it look identical: both
paid 0. Untangling which of your moves actually caused the ending is the central difficulty of the
whole field, and it has a name: the **credit assignment problem**.

> ⚠️ **The reward is a design decision, not a fact.** We could have paid −0.05 per turn to punish
> waffling, or +2 for an offer obtained honestly. We chose ±1 at the end and nothing in between,
> and every behaviour you see for the rest of this notebook is a consequence of that choice.

## 1.3 · 🕹️ Before any theory — go and get rejected a few times

Below is the actual interview. Click an action, watch what it does to them, and keep going until
one of the two endings. Then hit **New costume, new interview** and do it again.

Play for **two or three minutes**. Some things worth noticing while you do:

- the same action does **not** always produce the same reaction;
- some moves are clearly better in some situations than in others;
- you have started, without being asked to, forming a rule of the form *“when they look like **this**,
  I should do **that**”*. Hold on to that sentence. It is the rest of the notebook.

In [ ]:
iv.interview_game()

## 1.4 · State — what you decide to look at

Here is the idea that trips up almost everyone, so we are going to spend real time on it:

> ### A state is not a fact about the world. It is a **choice** you make.

The interviewer has a face, hands, a pen, a mug, a mortgage, a bad knee and an opinion about your
shoes. None of that is “the state” until you decide it is. **The state is the summary of the
situation that your policy is allowed to look at** — and you are about to choose it.

Three things are visible across the desk.

In [ ]:
iv.what_you_can_see()

### The trade-off, stated plainly

- **Too little in the state** → situations that need *different* answers get lumped together, and no
  policy can tell them apart. You have made good behaviour literally unrepresentable.
- **Too much in the state** → every situation becomes rare. With the same number of interviews you
  now have five, or fifteen, or forty-five separate things to learn about, each seen a fraction as
  often. You have made good behaviour *unlearnable in your lifetime*.

This is the same bias–variance instinct you already have from supervised learning, wearing a hat.

### 🧠 Quick check — before you choose

In [ ]:
iv.mc_quiz("what_is_state")

### 🎯 Task 1 — choose your state

**This is the only modelling decision you make in this notebook, and it sets up everything that
follows.**

Pick which components you want to look at: any subset of `["face", "hands", "pen"]`. The notebook
follows your choice all the way to the end — including a scoreboard at the end of Part 3, where we
find out whether you chose well. *(`["face"]` is a reasonable first bet. You will get to test it.)*

Everything below the first line is plumbing: it turns what is visible into the state you just said
you wanted to see.

In [ ]:
MY_STATE = ["face"]        # 🎯 your modelling choice — any subset of ["face", "hands", "pen"]

def observe(obs):
    '''What is visible across the desk  ->  the state my policy is allowed to look at.'''
    if obs["done"]:
        return ("end", obs["face"])          # 🥳 hired or 💔 rejected — nothing left to decide
    return tuple(obs[f] for f in MY_STATE)

demo = {"face": 3, "hands": 0, "pen": 2, "turn": 4, "done": False}
print("everything visible :", demo)
print("your state         :", observe(demo), " →", iv.state_label(observe(demo), MY_STATE))

Here is what you just committed to. Every situation in the left-hand column below is one your
policy will have to have an opinion about — and only these.

In [ ]:
STATES = iv.enumerate_states(MY_STATE)     # every non-terminal state your choice creates
iv.state_space_card(MY_STATE)

## 1.5 · Transition — how the world answers back

Taking an action moves you: you were in state `s`, you did `a`, and now you are in state `s′`. The
rule that governs that move is the **transition function**, and it comes in two flavours:

$$\textbf{deterministic:}\quad s' = T(s,a) \qquad\qquad
\textbf{stochastic:}\quad s' \sim P(\,\cdot \mid s, a)$$

- **Deterministic** — the same move from the same situation *always* lands in the same place. Clean,
  predictable, and completely unlike a person.
- **Stochastic** — it lands *somewhere*, with probabilities. Your compliment usually warms them up.
  Usually. **This is the world we are in.**

Here is the same interview again, with a switch. Play a few moves, then press **↻ Replay the same
moves** a couple of times in each world.

In the deterministic world you will get the same story back, every single time. In the stochastic
one — the real one — the identical sequence of sentences produces a different interview.

In [ ]:
iv.interview_game(mode_switch=True, title="🎲 The same four moves, twice")

**Same behaviour, different outcome.** Nothing about your decision-making changed between those
replays; the world simply rolled differently. Carry that with you: from here on, **one interview is
never evidence.** We will always be averaging.

### Now the arrows — and the thing this notebook will not give you

Draw the picture and it is a graph: your states, with arrows for what each action might do.

In [ ]:
iv.transition_graph(MY_STATE)

> ⚠️ **That picture is a sketch, not the map.** Only a handful of arrows are drawn, and which
> ones is arbitrary — do not read anything into *which* states appear to lead to an ending. In
> reality every state has four actions leading almost anywhere, and from nearly all of them there is
> some chance of walking straight into a “you are taken” or a “we will be in touch”. Drawing the
> whole thing would give you a solid black rectangle.

Every one of those arrows needs a number, and **you do not have a single one of them.** Nobody
handed you `P(s′ | s, a)` for a human being. You cannot look it up, and you certainly cannot ask.

In [ ]:
iv.model_free_card()

> 🧩 **The word from the lecture.** If we *did* know every arrow, and if the next state depended
> only on the current state and action, the whole thing would be a **Markov Decision Process** — and
> barely a learning problem at all: with the full table of probabilities in front of you, the best
> possible policy can simply be **calculated**, at your desk, without ever entering the room. That is
> the world of *planning*. We are in the other one.

### 🧠 Quick check — what is actually missing?

In [ ]:
iv.mc_quiz("modelfree")

## 1.6 · Policy — the rule you act by

The **policy** `π` is the object we are here to learn: the rule that turns a state into an action.

$$\textbf{deterministic:}\quad a = \pi(s) \qquad\qquad
\textbf{stochastic:}\quad a \sim \pi(a \mid s)$$

In this notebook we will end up with a deterministic one — a small table, *interviewer looks like
this → say that*. But everything starts from a policy that knows nothing at all.

### 🎯 Task 2 — run one interview under a policy

This loop is the shape of every RL program ever written, so it is worth typing once by hand:

**look → decide → act → record → repeat.**

`env.reset()` sits you down and returns the first observation. `env.step(a)` says the thing and
returns `(next observation, reward, done)`. Your `observe` from Task 1 is what turns those
observations into states.

> 💡 `policy` is passed in as a **function**: you call it, `policy(...)`, and it hands you back an
> action. The only question is what you have to give it.

In [ ]:
env = iv.Interviewer(seed=0)

def random_policy(state):
    '''Knows nothing, says whatever. Our starting point.'''
    return int(np.random.randint(N_ACTIONS))

def run_interview(policy, env=env):
    '''One full interview. Returns (states, actions, rewards).'''
    obs   = env.reset()
    state = observe(obs)
    states, actions, rewards = [], [], []
    done = False
    while not done:
        action = ???                      # 🎯 `policy` is a function — call it. On what?
        obs, reward, done = ???           # 🎯 say it, and see what the world does about it
        states.append(state); actions.append(action); rewards.append(reward)
        state = observe(obs)              # ← tomorrow's starting point
    return states, actions, rewards

# --- watch two interviews under the same (random) policy
np.random.seed(1)
for run in range(2):
    s, a, r = run_interview(random_policy)
    print(f"interview {run+1}: {len(a):2d} turns, "
          f"{'🥳 HIRED' if r[-1] > 0 else '💔 rejected'}, rewards = {[int(x) for x in r]}")

Look at those reward lists: `[0, 0, 0, 0, 0, 0, -1]`. That is the sparse-reward problem in
print. Six moves that paid nothing, and one number at the end that has to explain all of them.

That sequence of *(state, action, reward)* triples has a name: a **trajectory**, or an **episode**,
written **τ**. Everything from here on is about turning a pile of trajectories into a better policy.

---
✅ **That is the five-tuple.** State, action, transition, reward, policy — on one interview, in about
twenty minutes. The rest of the notebook answers one question: **which policy is the best one, and
how would we ever find it?**

---
# Part 2 — What “good” actually means

We have a policy that acts. We have no way of saying whether it acts *well*. Three quantities fix
that, and they are the same idea at three distances: **G**, **V** and **Q**. Getting comfortable
moving between them is the single most useful thing you can take out of this hour.

## 2.1 · The return — scoring one interview

A reward is what one turn paid. We don't want to maximise a turn, we want to maximise the whole
interview. The standard way to add an episode up is the **discounted return**:

$$G(\tau) \;=\; r_0 \;+\; \gamma\, r_1 \;+\; \gamma^2 r_2 \;+\;\dots\;=\; \sum_{t=0}^{T-1} \gamma^{t}\, r_t
\qquad \gamma \in [0, 1]$$

The **discount factor γ** is a dial for short-sightedness: at γ = 0 only this turn exists, at γ = 1 a
reward twenty turns from now counts exactly as much as one right here.

**And in our interview something nice happens.** Every reward is 0 except the last one, so that whole
sum collapses to a single term: an offer on turn `k` is worth exactly `γ^(k-1)`.

In [ ]:
iv.gamma_ladder()

Play with that slider until the point lands, because it is a point about *management*, not
about maths:

> **Nowhere in the reward does it say “be quick”.** Waiting costs nothing — the reward for a turn of
> waffling is 0. And yet, with γ < 1, an offer eight turns from now is worth barely half an offer
> right now, so an agent maximising G will hurry. **Speed was never in the reward. It was smuggled
> in by γ**, which we picked.

That is the shape of almost every RL horror story you will read: the objective contained something
nobody meant to put there, and the agent found it.

### 🎯 Task 3 — the discounted return, in code

Three lines, and you will reuse it in every remaining part of the notebook.

In [ ]:
GAMMA = 0.9        # you would like the offer today rather than in twenty minutes

def discounted_return(rewards, gamma=GAMMA):
    '''G = r_0 + gamma*r_1 + gamma^2*r_2 + ...'''
    total = 0.0
    for t, r in enumerate(rewards):
        total += ???           # 🎯 this turn's reward, weighted by how far in the future it is
    return total

# --- self-checks you can verify in your head
assert abs(discounted_return([0, 0, 0, 1.0]) - 0.729) < 1e-9, "γ³ = 0.729 — only the last term lives."
assert abs(discounted_return([1.0], 0.5) - 1.0)     < 1e-9, "The FIRST reward is never discounted."
assert abs(discounted_return([0, 0, -1.0], 1.0) + 1.0) < 1e-9, "γ = 1 is just the plain sum."
print("offer on turn 1 :", discounted_return([1.0]))
print("offer on turn 5 :", round(discounted_return([0, 0, 0, 0, 1.0]), 3), " ← same offer, 34% less value")

### 🔢 Predict before you compute

In [ ]:
iv.number_quiz("returns")

## 2.2 · From one interview to the worth of a situation

`G` describes one interview. Run the *same* policy again and you get a different `G`, because the
interviewer rolls dice. So `G` on its own is not a description of anything you can act on.

Average it, and it becomes one:

$$V^{\pi}(s) \;=\; \mathbb{E}_{\pi}\big[\,G \;\big|\; \text{starting from } s\,\big]$$

**The value of a state**: how much a situation is worth, if you behave as π from there on. Not what
happened once — what happens *on average*.

And it matches an instinct you already have. Walk into the room and they are already 😄 delighted,
and you have not opened your mouth yet: that interview is worth a lot. Walk in to 😠 eyebrows
furrowed, and — before you have done anything at all — you are in trouble.

### 🎯 Task 4 — measure it, the brute-force way

There is a completely obvious way to estimate an expectation: **run the thing many times and take
the average.** Run a few thousand interviews under the random policy, and for each one record two
things — the state you saw on turn 1, and the return G that interview eventually produced. Group by
the first, average the second, and you have V.

(This is **Monte-Carlo** value estimation, and it works precisely because we have a fresh costume.)

> 💡 `run_interview` already hands you `states` and `rewards`; the state you saw on turn 1 is just
> the first entry of one of them, and the return of the whole episode is Task 3 applied to the other.

In [ ]:
from collections import defaultdict

def estimate_V(policy, n_interviews=4000):
    '''V^pi(s) ~= average return of the interviews that STARTED in s.'''
    returns = defaultdict(list)
    for _ in range(n_interviews):
        states, actions, rewards = run_interview(policy)
        first_state = ???                    # 🎯 the situation you were in on turn 1
        G           = ???                    # 🎯 what that whole interview ended up being worth —
                                             #    you wrote the function for this in Task 3
        returns[first_state].append(G)
    return {s: float(np.mean(gs)) for s, gs in returns.items()}

np.random.seed(0)
V_random = estimate_V(random_policy)
iv.v_bars(V_random, MY_STATE,
          title="V(s) under the random policy — what each situation is worth if you flail")

Two things to take from that chart.

1. **The numbers are not equal.** Some situations are simply worth more than others, before you
   decide anything. That is what a value function *is*.
2. **These are the values of flailing.** `V_random` is not “the value of the state” in the abstract —
   there is no such thing. It is the value of that state *given how we behave*. Change the policy and
   every number moves. Value functions always carry a policy in their superscript: `V^π`.

> ⚠️ Also notice what this method needed: **complete interviews.** Monte-Carlo cannot tell you
> anything until the episode is over, because it has to know how the story ended. Remember that
> — we are going to fix it in Part 3.

## 2.3 · From the worth of a situation to the worth of a *move*

Here is the awkward thing about V: **it does not tell you what to do.**

`V^π(😄 delighted) = 0.42` is a lovely fact and it does not help you choose a sentence. Values are
attached to situations; decisions are made over actions. What we actually want is the same average,
but with the first move pinned down — and that is the **action-value function**, `Q`.

In [ ]:
iv.g_to_v_to_q()

Read those three identities out loud once; they are the mental furniture for the entire week.

- `V^π(s) = Σ_a π(a|s) Q^π(s,a)` — **V is Q, averaged over what you actually play.**
- `V*(s) = max_a Q*(s,a)` — **if you play optimally, a situation is worth its best move.**
- `π*(s) = argmax_a Q*(s,a)` — **and the optimal policy is “read the row, take the biggest number”.**

That last line is worth staring at. It says that if somebody handed you `Q*`, you would be *finished*
— no planning, no search, no model of the interviewer. Just a table lookup.

### 🔢 Check you can move between them

In [ ]:
iv.number_quiz("vq")

## 2.4 · The tempting shortcut, and why it fails here

At this point a very reasonable thought appears, and it is worth taking seriously because it is
*almost* right:

> *“In the lecture we saw that an optimal value function induces an optimal policy. So why not just
> learn `V*`, and read the policy off it?”*

It does induce one. Here it is:

$$\pi^*(s) \;=\; \arg\max_{a} \; \underbrace{\sum_{s'} P(s' \mid s, a)}_{\text{the interviewer's brain}}
\Big[\, R(s,a,s') \;+\; \gamma\, V^*(s')\,\Big]$$

Now read what that instruction is actually asking you to do. *For each of your four sentences, work
out where it would leave them, and average the value of those situations by how likely each one is.*
You cannot do the first half of that. **`P(s′|s,a)` is the one thing you were never given.**

And that is not a problem at learning time — it is a problem **at decision time**. Even if a fairy
handed you a perfect `V*` for free, right now, you would still be sitting in that chair unable to
choose a sentence.

Compare it with the same instruction written in terms of `Q*`:

$$\pi^*(s) \;=\; \arg\max_{a}\, Q^*(s, a)$$

**Read the row. Take the biggest number.** No sum, no `P`, nothing about where anything leads.

In [ ]:
iv.the_wall()

So we learn **Q** instead of **V** — not because Q is more elegant, but because Q has already
done the one thing we cannot do for ourselves: it has split the future up by action.

### 🧠 Return, V and Q — one pass

In [ ]:
iv.true_false_quiz("values")

---
# Part 3 — Q-learning

## 3.1 · Stop waiting for the end

Our Monte-Carlo estimate in Task 4 had a problem you probably felt: it could not say a single word
about a move until the whole interview was over. Every turn of every interview sat there, unused,
until somebody said “you are taken” or “we will be in touch”.

The **temporal-difference** idea from the lecture removes that wait. Instead of comparing your
estimate against a *completed* return, compare it against **one real step plus your own estimate of
the rest**:

$$\underbrace{V(s)}_{\text{what I thought}} \;\leftarrow\; V(s) \;+\; \alpha\Big[\;
\underbrace{r + \gamma V(s')}_{\text{what one step suggests}} \;-\; V(s)\;\Big]$$

You take one turn, see what actually happened, and drag your old opinion a fraction `α` of the way
towards the new one. This is *bootstrapping* — using an estimate to improve an estimate — and it
sounds like it should not work. It does, and it is the engine of the entire field.

**Now do the same thing to Q instead of V**, and use the identity from §2.3 —
`V*(s′) = max_a′ Q*(s′, a′)` — for the “rest of the story” term:

$$Q(s,a) \;\leftarrow\; Q(s,a) \;+\; \alpha\Big[\; r + \gamma \max_{a'} Q(s', a') \;-\; Q(s,a)\;\Big]$$

That single line is **Q-learning**. It is a direct translation of the Bellman optimality equation
into a rule you can run inside a live interview.

In [ ]:
iv.td_diagram()

### 🎯 Task 5 — one Q-learning update

Two lines. Read the diagram above as you write them.

> 💡 **How the table is shaped.** `Q[s]` is a row of four numbers, one per action — so `Q[s][a]` is
> what we currently think move `a` is worth in situation `s`, and `np.max(Q[s])` is the value of the
> best move available there.
>
> 💡 **About that `0.0 if done`.** It is already written for you, but it is worth knowing why it is
> there: when the interview has just ended there is no next situation at all, so there is no future
> value to add — the target is nothing but the reward you were handed.

In [ ]:
def q_learning_update(Q, s, a, r, s_next, done, alpha, gamma=GAMMA):
    '''Nudge Q(s, a) towards what this one turn suggests it should be. Returns the TD error.'''
    best_next = 0.0 if done else ???     # 🎯 how good is where we landed, if we act well from there?
    target    = ???                      # 🎯 what this turn now says the move (s, a) was worth
    td_error  = target - Q[s][a]         # how wrong we were
    Q[s][a]  += alpha * td_error         # move a fraction alpha of the way there
    return td_error

# --- self-check on a table we can read by hand
demo_Q = defaultdict(lambda: np.zeros(N_ACTIONS))
demo_Q[("A",)] = np.array([0.00, 0.50, 0.00, 0.00])   # we currently rate action 1 here at 0.50
demo_Q[("B",)] = np.array([0.10, 0.40, 0.30, 0.20])   # and the best move over in B at 0.40

err = q_learning_update(demo_Q, ("A",), 1, r=0.0, s_next=("B",), done=False, alpha=0.5)
print(f"target = 0 + 0.9*0.40 = {0.9*0.4:.2f}   ·   we said 0.50   ·   TD error = {err:+.2f}")
print("Q(A, 1) after a half-step:", round(demo_Q[("A",)][1], 3))
assert abs(err + 0.14) < 1e-9 and abs(demo_Q[("A",)][1] - 0.43) < 1e-9

err = q_learning_update(demo_Q, ("A",), 0, r=1.0, s_next=("end", 5), done=True, alpha=0.5)
print(f"\nAnd on the turn they hire you, the future is worth nothing: target = 1.0, "
      f"TD error = {err:+.2f}")
assert abs(err - 1.0) < 1e-9, "When done=True the target must be exactly r."
print("\n✅ That is Q-learning. The rest is a for-loop.")

## 3.2 · You have to try things you currently think are bad

There is one gap left. The update improves `Q` for moves we actually *make* — so if we always make
the move the table currently likes best, we will never find out about the other three. Our very
first, entirely uninformed guess would become permanent policy.

The standard fix is embarrassingly simple and works remarkably well: **ε-greedy**. With probability
`ε`, ignore the table and say something at random. Otherwise, take the best-looking move.

`ε` starts high — you know nothing, so wander — and decays as the table fills in.

Nothing to write here — but read it, because the second line is the only place in the whole
algorithm where the agent decides anything.

In [ ]:
def choose_action(Q, state, epsilon, rng):
    '''Explore with probability epsilon; otherwise trust the table.'''
    if rng.random() < epsilon:
        return int(rng.integers(N_ACTIONS))     # 🎲 try something, anything
    return int(np.argmax(Q[state]))             # 🏆 exploit: the move this row rates highest

# with epsilon = 0 the choice is forced; with epsilon = 1 nothing is
rng = np.random.default_rng(0)
demo_Q[("A",)] = np.array([0.1, 0.9, 0.2, 0.3])
print("epsilon = 0  →  always", ACTIONS[choose_action(demo_Q, ("A",), 0.0, rng)])
print("epsilon = 1  →  samples all four:",
      sorted(set(choose_action(demo_Q, ("A",), 1.0, rng) for _ in range(400))))

### 🤔 A confession about that `max`

We slipped something past you in Task 5, and this is the moment to own up to it.

The equation we are chasing, `Q*(s,a) = 𝔼[ r + γ·max_a′ Q*(s′,a′) ]`, describes an agent that plays
**optimally** from the next turn onwards — that is what the `max` is asserting. But we do not have
the optimal policy; finding it is the entire point of the exercise. And we certainly were not
playing it while collecting the data — we were saying random sentences to a stranger.

So the target is a statement about a policy we do not have, computed from experience produced by a
policy we would rather not keep. It sounds like it should not work.

It does, and the reason is worth naming. The `max` is a *claim about the future*, not a description
of what we did. Every time one entry of the table gets a little more accurate, the `max` over that
row does too, and the claim becomes a little less of a lie — everywhere at once. As long as we keep
visiting every situation and trying every move, the table converges on `Q*` **regardless of how
clumsily we behaved while gathering the experience**. That is what the exploration noise is buying:
not better behaviour, but the coverage that makes the claim safe.

Learning about one policy while following another is called **off-policy** learning, and it is a
real luxury — Q-learning gets to learn the best possible behaviour without ever performing it.
Notebook 02 gives that luxury up, and you will feel the difference immediately.

## 3.3 · The loop

Nothing new here — this cell just wires your two functions into the shape from §1.6, and keeps some
statistics so we can watch the improvement. Read it once; it is eleven lines of actual algorithm.

In [ ]:
def train(n_interviews=8000, gamma=GAMMA, seed=0):
    '''Q-learning. Returns the table and a log of how the interviews went.'''
    env = iv.Interviewer(seed=seed)
    rng = np.random.default_rng(seed + 1)
    Q   = defaultdict(lambda: np.zeros(N_ACTIONS))     # every move starts out worth exactly nothing
    hires, returns, turns = [], [], []

    for i in range(n_interviews):
        frac    = i / n_interviews
        epsilon = max(0.10, 1.0 - 2 * frac)            # wander early, commit later
        alpha   = 0.20 * (0.02 / 0.20) ** frac         # big steps early, careful ones later

        obs, done, rewards = env.reset(), False, []
        state = observe(obs)
        while not done:
            action            = choose_action(Q, state, epsilon, rng)
            obs, reward, done = env.step(action)
            next_state        = observe(obs)
            q_learning_update(Q, state, action, reward, next_state, done, alpha, gamma)
            state = next_state
            rewards.append(reward)

        hires.append(rewards[-1] > 0)
        returns.append(discounted_return(rewards, gamma))
        turns.append(len(rewards))
    return Q, hires, returns, turns

print(f"Sitting through 8000 interviews with the state {MY_STATE} …")
Q, hires, returns, turns = train()
print(f"done. Offer rate over the last 500: {np.mean(hires[-500:]):.1%}"
      f"   ·   average length: {np.mean(turns[-500:]):.1f} turns")

In [ ]:
iv.training_curve(hires, returns, turns=turns)

Three views of the same improvement. The offer rate climbs, the discounted return climbs
faster — and the interviews get **shorter**, which nothing in the reward ever asked for. That is γ,
quietly doing what we predicted in §2.1.

The early part of each curve is deliberately bad: ε is near 1, so the agent is essentially the random
policy from Part 1. Everything after that is the table filling in.

## 3.4 · What did it actually learn?

Every number in this table was estimated from nothing but interviews. No probabilities were ever
written down.

In [ ]:
iv.q_table_view(Q, MY_STATE)

Read a row: four numbers, one per thing you could say, and the ringed one is the move this
situation prefers. The `V(s)` column on the right is `max_a Q(s,a)` — the identity from §2.3, now
just a column of a spreadsheet.

## 3.5 · The deliverable

`π*(s) = argmax_a Q*(s,a)` — the identity from §2.3 — is one line of code. Hand the resulting policy
to the `run_interview` function you wrote in Task 2 and it plays for real, with no new machinery.

In [ ]:
def greedy_policy(Q):
    '''Turn a table of action-values into a rule for acting: read the row, take the best.'''
    def pi(state):
        return int(np.argmax(Q[state]))
    return pi

def evaluate(policy, n_interviews=2000, gamma=GAMMA, seed=99):
    '''Run a policy for real, and report offer rate, length and average return.'''
    test_env = iv.Interviewer(seed=seed)
    offers, lengths, Gs = [], [], []
    for _ in range(n_interviews):
        states, actions, rewards = run_interview(policy, env=test_env)
        offers.append(rewards[-1] > 0); lengths.append(len(rewards))
        Gs.append(discounted_return(rewards, gamma))
    return float(np.mean(offers)), float(np.mean(lengths)), float(np.mean(Gs))

offer, length, G       = evaluate(greedy_policy(Q))
offer_r, length_r, G_r = evaluate(random_policy)       # the flailing from Part 1, for scale
print(f"🎓 trained policy : {offer:6.1%} offers · {length:5.2f} turns · G = {G:+.3f}")
print(f"🎲 random policy  : {offer_r:6.1%} offers · {length_r:5.2f} turns · G = {G_r:+.3f}")

And here it is as the thing you actually walk into the room with.

In [ ]:
iv.policy_card(Q, MY_STATE)

### 🧠 One pass on the algorithm itself

In [ ]:
iv.true_false_quiz("qlearning")

## 3.6 · So — was your state a good choice?

We promised in §1.4 that the choice of state was yours and that we would come back to judge it. Here
is the honest test, and the only one available in real life: **hold everything else fixed and compare
results.** Same algorithm, same number of interviews, same interviewer. Only the definition of
“the situation I am in” changes.

`observe` reads the global `MY_STATE`, so re-pointing that variable is all it takes to retrain the
whole thing under a different modelling choice.

> ⏳ Six trainings — about half a minute.

In [ ]:
MY_CHOICE  = list(MY_STATE)          # remember what you picked, we put it back afterwards
CANDIDATES = [[], ["pen"], ["hands"], ["face"], ["face", "hands"], ["face", "hands", "pen"]]
if MY_CHOICE not in CANDIDATES:
    CANDIDATES.append(MY_CHOICE)

scores = {}
for choice in CANDIDATES:
    MY_STATE = choice                                    # ← observe() picks this up
    Q_c, _, _, _ = train(n_interviews=8000)
    label = (" + ".join(choice) if choice else "nothing at all")
    n_states = len(iv.enumerate_states(choice))
    offer_c, length_c, G_c = evaluate(greedy_policy(Q_c))
    scores[f"{label}  ({n_states} states)"] = (offer_c, length_c, G_c)
    print(f"  {label:24s} → G = {G_c:+.3f}   ({offer_c:.0%} offers, {length_c:.1f} turns)")

MY_STATE = MY_CHOICE                                     # put your choice back
iv.scoreboard(scores)

**Everything worth knowing about state design is in that chart.**

- **The face wins, and it is not close.** Five situations, each visited constantly, each genuinely
  predicting what happens next.
- **Adding things on top of the face makes it worse.** Not because the extra information hurts —
  strictly more information cannot hurt an *idealised* agent — but because 15 or 45 rows have to be
  learned from the same 8000 interviews that comfortably filled 5. **You paid for detail with data
  you did not have.**
- **Looking at nothing at all** does not collapse to zero, and the reason is worth understanding:
  with a single row the agent learns one safe sentence and repeats it, for eighteen turns, until
  somebody caves. It wins by attrition rather than judgement — which is exactly what γ punishes.
- **And here is the sharp one: the pen and the hands score *below* looking at nothing.** Not equal
  to it — below it. A state that splits the world along the wrong seam is worse than one that does
  not split it at all, because it makes the agent act *confidently* on a distinction that has no
  bearing on the decision. The pen is pure noise, so the three rows it creates disagree at random
  and the policy becomes unstable from run to run. The hands are more interesting: they carry real
  signal, and it is still not enough. Two interviewers with identical hands can be in opposite
  moods and need opposite sentences, so each row averages situations that wanted different answers,
  and the average is a compromise that suits neither.

> 💼 **The managerial version.** “Let's feed the model everything we have” is not a free choice, and
> neither is “let's just use the one signal we happen to log”. Every feature you add splits your
> data; every feature you leave out merges situations that may deserve different answers. The right
> state is the smallest one that still separates the cases that need separating — and finding it is
> modelling work, not tuning.

---
# Part 4 — 🏁 The final round

You got the job. Congratulations — it turns out there is a **second interview**, upstairs, with
somebody else entirely.

Here is the twist, and it is the last idea in the notebook: **the sheet you just learned is worthless
in that room.** This is a different person, with different dynamics. Flattery reads as transparent to
them. The technical jargon that was suicide downstairs is what earns their patience. Four of your
five lines are now wrong, and playing the old sheet here scores *worse than saying random things*.

So both of you start from nothing:

- **On the left, you.** You get to think, and you get to bring everything you know about people.
- **On the right, a Q-learning agent** with a **blank table**, running the exact algorithm you wrote
  in Task 5, learning in your browser as you watch. It has no priors, no transfer, no idea what a
  job is. It is also **not going to wait for you** — every second you spend deciding, it is
  finishing another interview and updating another row.

**The race:** first to average **G ≥ 0.35** over their **last 5 interviews**. That means offers, and
it means quick ones — the objective from §2.1, applied to both of you equally.

> 🎯 This is genuinely winnable, and it is winnable for a reason worth noticing: you can form a
> hypothesis about a person after two interviews. The agent needs dozens. What it has instead is
> that it never gets tired, never gets embarrassed, and never stops.

In [ ]:
iv.race()

However that went, look at what just happened on the right-hand side.

That agent knew nothing about interviews, people, or language. It ran the update you wrote in
Task 5 — `Q ← Q + α·(target − Q)` — a few hundred times, and behaviour appeared. No model of the
interviewer was ever built. Nobody told it which sentence was correct. It was told `+1` or `−1` at
the end of some conversations, and that was enough.

That is the whole trick, and it is the same trick all the way up: **AlphaGo, robot locomotion, and
the reinforcement-learning stage of every modern chat model are this loop with a bigger table.**

And the trap you just walked into is worth as much as the trick. A policy is not knowledge about
interviews — **it is knowledge about one environment**, and it silently stops being true when the
environment changes. Your model does not tell you when that has happened. The interviewer does.

---
# 🎓 Wrap-up

| The idea | In our interview | In one line |
|---|---|---|
| **State** `s` | what you chose to look at | a modelling decision, not a fact |
| **Action** `a` | smile / jargon / compliment / agree | the closed list of what you can do |
| **Transition** `P(s′\|s,a)` | how they react | the world's business — and we never saw it |
| **Reward** `r` | +1 hired, −1 rejected, 0 in between | what counts as success, and *you* wrote it |
| **Policy** `π(s) → a` | your final sheet | the thing being learned |
| **Return** `G(τ)` | `γ^(k−1)·(±1)` | how one interview went |
| **Value** `V^π(s)` | `𝔼[G \| s]` | how good a *situation* is |
| **Action-value** `Q^π(s,a)` | `𝔼[G \| s, a]` | how good a *move* is |
| **Bellman optimality** | `Q*(s,a) = 𝔼[r + γ max Q*(s′,·)]` | today's value, in terms of tomorrow's |
| **TD update** | `Q ← Q + α·(target − Q)` | learn from one turn, not one interview |
| **Q-learning** | act ε-greedy, update towards the max | the loop that produced your table |

**The six things worth carrying out of here**

1. **The state is your decision, and it is the expensive one.** Too little and good behaviour is
   unrepresentable; too much and it is unlearnable. The scoreboard in §3.6 is what that trade-off
   costs in practice.
2. **γ is part of the objective, not of the world.** Nothing rewarded speed, and the agent got fast
   anyway. Every quantity you put in the objective will be pursued, including the ones you did not
   realise you put there.
3. **`V` scores situations, `Q` scores moves — and only one of them lets you act without a model.**
   Being able to slide between `G`, `V` and `Q` in either direction is most of the fluency you need
   for the rest of the week.
4. **Bootstrapping is the engine.** Improving an estimate using another estimate, one turn at a time,
   is what turns a calculation that needs the interviewer's brain into a method that needs only
   the interviewer.
5. **Nothing here ever knew the transition probabilities.** Not while learning, not while acting.
   That is what model-free buys you, and it is why the same ideas survive when the state is a
   conversation instead of a face.
6. **A policy is knowledge about an environment, not about the world.** Part 4 is the cheapest
   version of the most expensive lesson in applied RL: the sheet stays confident, and stops being
   true, the moment the environment moves. Nothing in the model raises its hand to tell you.

### The crack this notebook leaves open

Look once more at how we chose an action: `argmax` over a row of `Q`. It works beautifully here,
because there are four actions and forty rows. Now imagine the action is not *one of four sentences*
but *a torque on eleven joints*, or *the next token out of a vocabulary of 100 000*. The `argmax`
becomes the hard part, then the impossible part.

So: what if we skipped the value table entirely and **learned the policy directly**?

- **Notebook 02 — policy gradients:** exactly that. No table of values, no argmax; a policy that is
  differentiated and pushed towards what worked.
- **Notebook 03 — actor–critic:** the discovery that you wanted the value function back after all,
  as a *critic*.
- **Notebook 04 — GAE & PPO:** making that stable enough to trust.
- **Notebook 05 — RL for LLMs:** the same loop, where the interview is a generated answer.

---
## 🏁 Final boss — clear the notebook
Everything you just learned, one statement at a time: **state, actions, transitions, γ, V, Q,
Bellman, TD, exploration.** The rules: **3 lives**, **10 seconds** per question, and a wrong answer
*or* a timeout costs a life. Reach **10 correct** to pass.

You have done harder interviews today. 🍀

In [ ]:
iv.flash_quiz()